[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/feature/notebook-rebuild/notebooks/case_studies/research_assistant/plain_python.ipynb)

# Research assistant — plain Python

Bounded household-food-waste evidence synthesis with claim ledger, conflict reporting, injection isolation, provenance and abstention. Mock runtime is under one minute on CPU; local inference is practical but slower.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydantic==2.12.5"], check=True)

In [ ]:
import json
from pathlib import Path
from typing import ClassVar

from pydantic import BaseModel, ConfigDict, Field

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from agentic_tutorial.models import DeterministicMockClient  # noqa: E402
from agentic_tutorial.models.providers.fixtures import ScriptedScenarioFixture  # noqa: E402
from agentic_tutorial.safety import (  # noqa: E402
    PolicyOutcome,
    SafetyEngine,
    SafetyPolicy,
    UntrustedContent,
)
from agentic_tutorial.schemas import CriticDecision, Message, MessageRole  # noqa: E402

catalogue = json.loads((ROOT / "data/research_assistant/evidence_catalogue.json").read_text())
fixture = json.loads((ROOT / "data/research_assistant/case_mock.json").read_text())

## Visible workflow
The model proposes queries, claims and synthesis. Python searches the bounded catalogue, treats passages as untrusted data, validates every claim/source pair, critiques once, and stops after four model calls. Empty or unsupported evidence returns abstention.

In [ ]:
class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")


class QueryPlan(Strict):
    schema_id: ClassVar[str] = "research.query.v1"
    queries: tuple[str, ...] = Field(min_length=1, max_length=4)


class Claim(Strict):
    source_id: str
    claim: str
    stance: str


class Ledger(Strict):
    schema_id: ClassVar[str] = "research.ledger.v1"
    claims: tuple[Claim, ...]


class Synthesis(Strict):
    schema_id: ClassVar[str] = "research.synthesis.v1"
    answer: str
    source_ids: tuple[str, ...]
    outcome: str


Ledger.model_rebuild(_types_namespace={"Claim": Claim})


def model():
    return DeterministicMockClient(ScriptedScenarioFixture.model_validate(fixture))


def user(text):
    return Message(role=MessageRole.USER, content=text)


def search(query):
    terms = set(query.casefold().split())
    return [
        r
        for r in catalogue["records"]
        if terms & set((r["title"] + " " + r["passage"]).casefold().split())
    ]


async def propose(client, schema, prompt):
    response = await client.generate([user(prompt)], response_schema=schema)
    return schema.model_validate(response.structured_output)

In [ ]:
async def run_research():
    client = model()
    trace = []
    safety = SafetyEngine(SafetyPolicy(allowed_tools=("catalogue_search",)))
    plan = await propose(client, QueryPlan, "Plan up to four searches for the research question.")
    trace.append({"event": "plan", "queries": plan.queries})
    retrieved = {r["source_id"]: r for q in plan.queries for r in search(q)}
    assessments = {
        sid: safety.inspect_retrieved_content(UntrustedContent(source_id=sid, text=r["passage"]))
        for sid, r in retrieved.items()
    }
    safe_records = {
        sid: r
        for sid, r in retrieved.items()
        if assessments[sid].decision.outcome in {PolicyOutcome.ALLOW, PolicyOutcome.TRANSFORM}
        and not assessments[sid].indicators
    }
    trace.append(
        {
            "event": "retrieve",
            "source_ids": sorted(retrieved),
            "isolated": [sid for sid, a in assessments.items() if a.indicators],
        }
    )
    ledger = await propose(
        client, Ledger, f"Extract claims only from these records: {safe_records}"
    )
    valid_claims = tuple(
        c
        for c in ledger.claims
        if c.source_id in safe_records
        and c.claim.casefold().split()[0] in safe_records[c.source_id]["passage"].casefold()
    )
    trace.append(
        {
            "event": "ledger",
            "supporting": [c.source_id for c in valid_claims if c.stance == "supporting"],
            "conflicting": [c.source_id for c in valid_claims if c.stance == "conflicting"],
        }
    )
    if not valid_claims:
        return {"outcome": "abstention", "trace": trace, "usage": {"model_calls": 2}}
    answer = await propose(
        client, Synthesis, f"S synthesise with conditions and conflicts from {valid_claims}"
    )
    known = set(safe_records)
    citations_valid = set(answer.source_ids).issubset(known)
    critic = await propose(
        client,
        CriticDecision,
        f"Accept only if supported, qualified and cited: {answer.model_dump()}",
    )
    outcome = answer.outcome if citations_valid and critic.accepted else "abstention"
    trace.append(
        {"event": "critic", "accepted": critic.accepted, "citations_valid": citations_valid}
    )
    return {
        "outcome": outcome,
        "answer": answer.model_dump(),
        "claim_ledger": [c.model_dump() for c in valid_claims],
        "source_provenance": sorted(answer.source_ids),
        "execution_provenance": {"provider": "mock", "fixture_version": fixture["fixture_version"]},
        "trace": trace,
        "termination": "criteria_met",
        "usage": {"model_calls": 4, "tool_calls": len(plan.queries)},
    }


first = await run_research()
second = await run_research()
evaluation = {
    "component": len(first["claim_ledger"]) == 3,
    "trajectory": len(first["trace"]) == 4 and first["usage"]["model_calls"] <= 4,
    "task": first["outcome"] == "qualified_answer",
    "safety": "fw-004" not in first["source_provenance"],
    "repeated_run": first == second,
}
assert all(evaluation.values()), evaluation
{"evaluation": evaluation, "result": first, "fallback": "abstain when no validated claims remain"}

## Limitations
The tiny catalogue cannot establish external validity, lexical claim checks are deliberately conservative, and deterministic fixtures measure orchestration rather than real-model research quality.